# Week 2 Exercises

## Introduction

As with last week, work through these exercise by adding your own code to the cells to get the desired output. Avoid using AI to complete the exercises, unless otherwise instructed to seek outside help.

<!-- Before beginning this week's exercises, there are two additional `.csv` files you should download. These are historic meteological station data for selected UK meteorological stations, as available from [the UK Met Office website](https://www.metoffice.gov.uk/research/climate/maps-and-data/historic-station-data). I have provided a clean dataset of both [the Heathrow met station](https://raw.githubusercontent.com/trchudley/geospatial-scientific-computing/tree/main/week/02/heathrow_met_data.csv) and the [Lerwick met station](https://raw.githubusercontent.com/trchudley/geospatial-scientific-computing/tree/main/week/02/lerwick_met_data.csv), available from the end links. -->

This exercise makes use of [a new package, `meteostat`](https://dev.meteostat.net/python), to download met station data. [Meteostat](https://meteostat.net/en/) provides open and free access to historical met station observations from a range of public agencies, including public agencies such as the UK Met Office. Unlike our previous packages, `meteostat` is not available from `conda-forge`, so we must use a different 'storefront' (PyPi, accessible through the `pip` command). Ensure you are in the correct environment (e.g. `conda activate gsc`) and download `meteostat` using the command:

```bash
pip install meteostat
```

You should now be able to import the necessary packages for these exercises:

In [ ]:
import datetime
import pandas as pd
import meteostat as ms
import matplotlib.pyplot as plt

from scipy import stats

## Getting new data from `meteostat`

We can use `meteostat` built-in functions to search for nearby met station records for a given latitude and longitude. Different weather stations will have different provenance, with varying historical length, recorded variables, and intermittency. We will search for London Heathrow Airport met station, which has a long and complete record.

In [ ]:
# latitude and longitude of Heathrow Airport
latitude = 51.47
longitude = -0.46

# Find the 4 nearest weather stations within a 50 km radius of Bristol
point = ms.Point(latitude, longitude)
stations = ms.stations.nearby(point, radius=50000, limit=4) 

# View dataframe
stations

,name,country,region,latitude,longitude,elevation,timezone,distance
id,,,,,,,,
03772,London Heathrow Airport,GB,ENG,51.4833,-0.4500,24,Europe/London,1633.0
03672,Northolt,GB,ENG,51.5500,-0.4167,38,Europe/London,9386.8
EGTI0,Leavesden / North Watford,GB,ENG,51.6833,-0.4167,102,Europe/London,23905.9
03779,London Weather Centre,GB,ENG,51.5167,-0.1167,5,Europe/London,24327.5


We can download `ms.daily()` or `ms.monthly()` records, supplying a station ID (from the table above) and a date range:

In [110]:
# the selected station ID
id = "03772"

# search for 30 years of data in the `datetime` datatype: we create this using the `datetime` standard library package
start = datetime.date(1996, 1, 1)
end = datetime.date(2025, 12, 31)

# stations is returns as a pandas dataframe, sorted by distance
ts = ms.monthly(ms.Station(id=id), start, end) # Get monthly data for the stations in the specified date range
heathrow_df = ts.fetch() # Fetch the data from the Meteostat API and return it as a pandas dataframe

# View dataframe
heathrow_df

,temp,tmin,tmax,txmn,txmx,prcp,pres,tsun
time,,,,,,,,
1996-01-01,5.2,3.1,7.2,<NA>,<NA>,42.0,1010.2,<NA>
1996-02-01,3.4,0.1,6.8,<NA>,<NA>,49.0,1013.3,<NA>
1996-03-01,5.3,2.1,8.6,<NA>,<NA>,31.0,1018.3,<NA>
1996-04-01,9.9,5.5,14.3,-1.5,23.5,25.0,1017.2,<NA>
1996-05-01,10.5,5.9,15.1,1.2,26.1,26.0,1014.5,<NA>
...,...,...,...,...,...,...,...,...
2025-08-01,19.6,14.1,25.0,9.4,33.3,23.0,1016.5,11520
2025-09-01,15.7,11.1,20.2,5.0,27.4,49.0,1014.9,10260
2025-10-01,12.9,10.0,15.7,3.9,21.3,53.0,1014.4,4260


This dataset is already cleaned, with a `DateTime` index showing the monthly data. This means we can begin doing some interesting things!

## Exercises

### Multi-Panel Plots

Create a figure with two subplots: the one on the left should show the `temp` (temperature) record, and the one on the right the `prcp` (precipiation) record. They should have proper labels on the x axis (Time) and y axis (Average Monthly Temperature in ˚C or Monthly Precipation in mm), and a figure title as appropriate. 

Feel free to have fun and experiment with figure colours and styles! However, when producing figures for other readers, you should consider making colour choices friendly to colour-blind readers (e.g. avoiding red-green colour pairs) - more information can be read about this [here](https://www.nature.com/articles/d41586-021-02696-z).

In [ ]:
# TODO: Plot the described figure here.

### Annual Averages

Use the `groupby` or `resample` tool to calculate:

1. The annual average temperature variables (`temp`, `tmin`, `tmax`)
2. The cumulative average rainfall varaible (`prcp`)

You will need different statistical summaries (`.average()` vs `.cumsum()`) to do this.

Make sure these new dataseries all exist within a single new dataframe, called `heathrow_annual_df`.

In [ ]:
# TODO: Produce a new dataframe `heathrow_annual_df`, with the average annual temperature and cumulative annual rainfall for each year in the dataset.

Produce the same multi-panel figure as above, but this overlay the annual average datasets to the monthly dataset (as we did for the Mauna Loa data). You may copy and past your multi-panel plot code from above and add the new plotting code.

In [86]:
# TODO: Monthly and annual temerature/precipitation data on one multi-panel plot.

Using the new annual data and `stats.linregress`, can you establish whether there has been a statistically significant increase in (i) annual average temperature and (ii) annual cumulative reinfall throught time?

> Hint 1: `stats.linregress` cannot accept the `datetime` objects you will get from `heathrow_annual_df.index`. You can retrieve the years as `int` values by providing `heathrow_annual_df.index.year`.

> Hint 2: `stats.linregress` cannot accept `NaN` data. Where there are holes in the dataset, you will need to filter out NaN data. `pandas` has a handy method to do this: `.dropna()`. For instance:

```python
prcp = heathrow_df['prcp'].dropna()
stats.linregress(prcp.index, prcp.values)
```

In [ ]:
# TODO: Check whether the average annual temperature has changed over the 30-year period. Use a linear regression to determine whether there is a statistically significant trend in the data.

In [ ]:
# TODO: Do the same for average cumulative rainfall.

Which of these results are significant? What do you think explains a positive (or null) result?

### Comparing datasets

Let's bring in a new dataset to compare: Leuchars, near St Andrews in Scotland, as `leuchars_df`:

In [122]:
ts = ms.monthly(ms.Station(id='03171'), start, end) # Get monthly data for the stations in the specified date range
leuchars_df = ts.fetch() # Fetch the data from the Meteostat API and return it as a pandas dataframe

Create comparitive plots that allow you to visualise the difference between the Heathrow and Leuchars datasets:

In [ ]:
# TODO: Construct a plot comparing the monthly average temperature and cumulative rainfall for Heathrow and Leuchars over the 30-year period.

Using the descriptive statistics tools and temporal indexing, answer the following questions. You may have to do some independent searching to discover additional filtering tools.

1. What is the average temperature and cumulative rainfall over the entire time period for Heathrow and Leuchars?

In [ ]:
# TODO: Explore differences between the Heathrow and Leuchars datasets.

2. Get similar average values, but for the ten-year periods 1996-2005, 2006-2015, and 2016-2025. Does there appear to be a trend in the difference between these values?

In [127]:
# TODO: Explore decadal averages.

3. Is there a greater difference average in temperature and rainfall between Heathrow and Leuchars in December or July?

In [ ]:
# TODO: Explore December and July differences

4. Is there a statistically significant correlation between the monthly (i) temperature; and (ii) rainfall in Heathrow and Leuchars?

In [139]:
# TODO: Assess whether temperature and rainfall are correlated between Heathrow and Leuchers.

## Extension Task / Homework: Your own data

I now want to provide you with some time and space to get to grips with exploring some datasets of your own interest.

You may wish to continue explore the `meteostat` tool above, but I would recommend trying to explore datasets of your own interest. Find tabular data (`.csv`, `.xlsx`) online, or feel free to go back to some data you previous have encountered (and are allowed to use). Some potential sources of data you could explore include:

 - [The UK National Data Library](https://www.data.gov.uk/).
 - [Bristol Council Open Data](https://opendata.bristol.gov.uk/).
 - [Our World in Data](https://ourworldindata.org/).
 - [Climate Change Knowledge Portal](https://climateknowledgeportal.worldbank.org/).
 - [UN Data](https://data.un.org/).
 - [European Environment Agency (EEA) Datahub](https://www.eea.europa.eu/en/datahub).

Trying to find suitable data to explore from public seaches is a key skill to develop!

Once you have found some data that interests you, try and interrogate the data in the ways you have been shown today:

 - The data you download may require cleaning and modifying before interrogation.
 - Create exploratory statistics, test trends and hypothesis, and construct plots to visualise data. 
 - Group the data by time periods or by other classifications to investigate change over time or between groups.
 - If you have an idea or question that is not covered by our introduction today, you may need to explore additional functionality of `pandas` and other tools.

When exploring 'real' data, the answers and solutions may fall outside what has been covered in the material today. Feel free to talk to us, explore online documentation and example material, and make use of Googling and even LLMs to work towards anything that might interest you. 

**As part of your homework, share a plot you have constructed on the Blackboard Week 2 discussion board! Provide some context for the data, and explain what your plot shows.**